# 03 — Feature Engineering

Notebook ini mengekstrak fitur-fitur prediktif dari data bersih untuk digunakan oleh model ML.

## Input
- `data/df_clean.csv` — keluaran dari `02_cleaning.ipynb` (1,047,855 baris × 12 kolom)

## Output
- `data/df_feat.csv` — data dengan 103 kolom fitur (1,047,135 baris)

## Fitur yang Dibuat
| Kategori | Contoh | Keterangan |
|---|---|---|
| Temporal dasar | `hour`, `day`, `month`, `dayofweek`, `quarter` | Dekomposisi waktu |
| Musim Bangladesh | `season` (winter/pre_monsoon/monsoon/post_monsoon) | Kalender monsoon |
| Cyclical encoding | `hour_sin/cos`, `month_sin/cos`, `dow_sin/cos` | Mencegah diskontinuitas di batas (jam 23→0) |
| Lag features | `aqi_lag1`, `pm10_lag24`, … | 1, 3, 6, 24 jam lalu per kota |
| Rolling features | `pm2_5_roll24m`, `aqi_roll6std`, … | Mean & std window 3/6/24 jam, dengan `shift(1)` sebelum rolling untuk mencegah lookahead |
| Interaksi polutan | `pm_ratio_lag1`, `oxidant_load_lag1`, `combustion_idx_lag1` | Berbasis lag-1 untuk mencegah target leakage |

**Prasyarat:** Jalankan `02_cleaning.ipynb` terlebih dahulu.
**Notebook berikutnya:** `04_preprocessing.ipynb`

### 5.1 Ekstraksi Fitur Temporal

In [1]:
# === SEL INISIALISASI ===
import pandas as pd
import numpy as np
import os

df_clean = pd.read_csv('data/df_clean.csv', parse_dates=['datetime'])
print(f"df_clean dimuat: {df_clean.shape}")

df_clean dimuat: (1047855, 12)


In [2]:
df_feat = df_clean.copy()

# Fitur waktu dasar
df_feat['hour']        = df_feat['datetime'].dt.hour
df_feat['day']         = df_feat['datetime'].dt.day
df_feat['month']       = df_feat['datetime'].dt.month
df_feat['year']        = df_feat['datetime'].dt.year
df_feat['dayofweek']   = df_feat['datetime'].dt.dayofweek   # 0=Senin
df_feat['dayofyear']   = df_feat['datetime'].dt.dayofyear
df_feat['quarter']     = df_feat['datetime'].dt.quarter
df_feat['weekofyear']  = df_feat['datetime'].dt.isocalendar().week.astype(int)
df_feat['is_weekend']  = (df_feat['dayofweek'] >= 5).astype(int)

# Bagian hari
def get_period(hour):
    if   0 <= hour < 6:  return 'night'
    elif 6 <= hour < 12: return 'morning'
    elif 12 <= hour < 18: return 'afternoon'
    else:                 return 'evening'

df_feat['period_of_day'] = df_feat['hour'].apply(get_period)

# Mapping musim Bangladesh (monsoon calendar)
def get_season_bangladesh(month):
    if month in [12, 1, 2]:   return 'winter'
    elif month in [3, 4, 5]:  return 'pre_monsoon'  # panas
    elif month in [6, 7, 8, 9]: return 'monsoon'    # hujan
    else:                      return 'post_monsoon' # sejuk

df_feat['season'] = df_feat['month'].apply(get_season_bangladesh)

# Cyclical encoding untuk hour dan month
df_feat['hour_sin']  = np.sin(2 * np.pi * df_feat['hour']  / 24)
df_feat['hour_cos']  = np.cos(2 * np.pi * df_feat['hour']  / 24)
df_feat['month_sin'] = np.sin(2 * np.pi * df_feat['month'] / 12)
df_feat['month_cos'] = np.cos(2 * np.pi * df_feat['month'] / 12)
df_feat['dow_sin']   = np.sin(2 * np.pi * df_feat['dayofweek'] / 7)
df_feat['dow_cos']   = np.cos(2 * np.pi * df_feat['dayofweek'] / 7)

new_temporal_cols = ['hour','day','month','year','dayofweek','dayofyear','quarter',
                     'weekofyear','is_weekend','period_of_day','season',
                     'hour_sin','hour_cos','month_sin','month_cos','dow_sin','dow_cos']
print(f'   Fitur baru: {new_temporal_cols}')

   Fitur baru: ['hour', 'day', 'month', 'year', 'dayofweek', 'dayofyear', 'quarter', 'weekofyear', 'is_weekend', 'period_of_day', 'season', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'dow_sin', 'dow_cos']


### 5.2 Fitur Lag & Rolling (Time-Series Features)

In [3]:
# Lag dan rolling window per kota
# Pakai groupby().shift() dan groupby().transform() agar city_id/city_name
# tidak terdrop — konsisten dengan src/features.py (pandas 3.x compatibility)

lag_features = ['pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide',
                'sulphur_dioxide', 'ozone', 'aqi']

# Pastikan urutan data sudah benar sebelum membuat fitur time-series
df_feat = df_feat.sort_values(['city_id', 'datetime']).reset_index(drop=True)

print('Running Lag and Rolling...')
for col in lag_features:
    if col not in df_feat.columns:
        continue
    # Lag features (1 jam, 3 jam, 6 jam, 24 jam lalu)
    for lag in [1, 3, 6, 24]:
        df_feat[f'{col}_lag{lag}'] = df_feat.groupby('city_id')[col].shift(lag)
    # Rolling mean dan std (shift(1) dulu untuk menghindari data leakage)
    for window in [3, 6, 24]:
        df_feat[f'{col}_roll{window}m'] = df_feat.groupby('city_id')[col].transform(
            lambda x: x.shift(1).rolling(window, min_periods=1).mean()
        )
        df_feat[f'{col}_roll{window}std'] = df_feat.groupby('city_id')[col].transform(
            lambda x: x.shift(1).rolling(window, min_periods=1).std()
        )

print(f'Total kolom sekarang: {df_feat.shape[1]}')

Running Lag and Rolling...
Total kolom sekarang: 99


### 5.3 Fitur Interaksi Polutan

In [4]:
# Rasio dan interaksi yang bermakna secara domain (berbasis lag1 untuk mencegah leakage)
#
# Definisi (konsisten dengan src/features.py):
#   pm_ratio_lag1     = pm2_5[t-1] / (pm10[t-1] + ε)   — fraksi partikel halus
#   pm_total_lag1     = pm10[t-1] + pm2_5[t-1]          — beban partikel total
#   oxidant_load_lag1 = NO2[t-1] + O3[t-1]              — beban oksidan
#   combustion_idx_lag1 = CO[t-1] / (NO2[t-1] + ε)     — indikator pembakaran tidak sempurna
#     (CO tinggi & NO2 rendah → pembakaran smoldering; CO rendah & NO2 tinggi → pembakaran sempurna)

eps = 1e-8
df_feat['pm_ratio_lag1']       = df_feat['pm2_5_lag1'] / (df_feat['pm10_lag1'] + eps)
df_feat['pm_total_lag1']       = df_feat['pm10_lag1'] + df_feat['pm2_5_lag1']
df_feat['oxidant_load_lag1']   = df_feat['nitrogen_dioxide_lag1'] + df_feat['ozone_lag1']
df_feat['combustion_idx_lag1'] = df_feat['carbon_monoxide_lag1'] / (df_feat['nitrogen_dioxide_lag1'] + eps)

print('Fitur interaksi polutan (lag1):')
for col in ['pm_ratio_lag1', 'pm_total_lag1', 'oxidant_load_lag1', 'combustion_idx_lag1']:
    print(f'  {col}: min={df_feat[col].min():.4f}, max={df_feat[col].max():.4f}')

Fitur interaksi polutan (lag1):
  pm_ratio_lag1: min=0.1218, max=4.5601
  pm_total_lag1: min=2.1000, max=646.9304
  oxidant_load_lag1: min=9.6000, max=267.3000
  combustion_idx_lag1: min=0.0102, max=66800000000.0000


### 5.4 Hapus Baris dengan Lag NaN & Drop Kolom Tidak Diperlukan

In [5]:
n_before = df_feat.shape[0]

# Lag terbesar adalah 24 jam, jadi 24 baris pertama per kota akan punya NaN
df_feat = df_feat.dropna(subset=[c for c in df_feat.columns if 'lag24' in c])
df_feat = df_feat.reset_index(drop=True)

print(f'Baris dengan lag NaN dihapus: {n_before - df_feat.shape[0]:,}')
print(f'Shape setelah: {df_feat.shape}')

Baris dengan lag NaN dihapus: 720
Shape setelah: (1047135, 103)


In [6]:
# === SIMPAN HASIL FEATURE ENGINEERING ===
os.makedirs('data', exist_ok=True)
df_feat.to_csv('data/df_feat.csv', index=False)
print(f"df_feat disimpan -> data/df_feat.csv  {df_feat.shape}")

df_feat disimpan -> data/df_feat.csv  (1047135, 103)


---
## 6. Preprocessing (Encoding + Scaling)